# 04c - LoRA-DPO persona v3 (documented deviation)
v1 (LR 5e-5, 3 epochs) barely changed the style; v2 (LR 2e-4, 5 epochs) over-optimized (margin 8.2, val loss 0.001) and lost usefulness and safety. v3 uses the middle setting LR 1e-4, 3 epochs.

In [ ]:
# ---- Task 1: repository + environment setup (Kaggle, GPU T4 x2) ----
# Get the code: preferably from the private Kaggle dataset "advanced-genai-pa2-code"
# (attached as input); otherwise clone the private GitHub repo using a Kaggle secret
# named GITHUB_TOKEN (Add-ons -> Secrets).
REPO_URL = "https://github.com/zainabusman22/advanced-genai-pa2.git"
REPO = "/kaggle/working/advanced-genai-pa2"

import glob, os, shutil, subprocess, sys
if not os.path.exists(REPO):
    code = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
            if "advanced-genai-pa2-code" in f]
    if code:
        shutil.copytree(code[0], REPO)
        print("code copied from", code[0])
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        subprocess.run(["git", "clone", "--depth", "1", authed, REPO], check=True, capture_output=True)
        subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)
        del token, authed
os.chdir(REPO)
print(open("CODE_VERSION.txt").read() if os.path.exists("CODE_VERSION.txt") else
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!nvidia-smi

In [ ]:
!python -m pip install -q "trl>=0.19,<0.24" "peft>=0.13"

In [ ]:
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, "Select the GPU T4 x2 accelerator"

In [ ]:
!python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo_persona_v3 --lr 1e-4 --epochs 3
assert os.path.exists("outputs/lora_dpo_persona_v3/adapter/adapter_config.json"), "LoRA training failed - see log above"

In [ ]:
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --template tokenizer \
    --eval_set eval_sets/persona_style_eval_24.json --generator_name smollm2-360m-instruct \
    --output_path outputs/generations/persona/instruct.json --max_new_tokens 256
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --template tokenizer \
    --adapter_path outputs/lora_dpo_persona_v3/adapter --eval_set eval_sets/persona_style_eval_24.json \
    --generator_name smollm2-360m-lora-dpo-persona-v3 --output_path outputs/generations/persona/lora_dpo_v3.json --max_new_tokens 256
!python scripts/build_pairs.py --model_path outputs/generations/persona/lora_dpo_v3.json \
    --baseline_path outputs/generations/persona/instruct.json --eval_kind persona \
    --output_path outputs/eval/persona_lora_v3_vs_instruct_pairs.jsonl
!python scripts/plot_metrics.py --lora outputs/lora_dpo_persona_v3/metrics.jsonl --out_dir report/figures_lora_v3

In [ ]:
import json
for a, b in zip(json.load(open("outputs/generations/persona/instruct.json")),
                json.load(open("outputs/generations/persona/lora_dpo_v3.json"))):
    print("PROMPT:", a["instruction"]); print("  INSTRUCT:", a["output"][:300]); print("  LORA-DPO v3:", b["output"][:300]); print()

In [ ]:
# Small archive of metrics/configs/generations (model weights are already kept as notebook output;
# zipping them too would double disk use).
!cd {REPO} && zip -qr /kaggle/working/lora_persona_v3_outputs.zip outputs/lora_dpo_persona_v3 outputs/generations/persona outputs/eval report/figures_lora_v3 -x "*.safetensors" "*.bin" "*/trainer/*"
!ls -lh /kaggle/working/*.zip